# Fase 3: Machine Learning (Aprendizaje No Supervisado)

En esta fase utilizaremos el algoritmo **K-Means** para descubrir patrones ocultos en el comportamiento eléctrico de Ecuador. 
El objetivo es agrupar las regiones/sectores según su eficiencia (relación entre la Demanda Máxima, la Energía Facturada y el Porcentaje de Pérdidas).

Utilizaremos el **Método del Codo** y el **Coeficiente de Silueta** para justificar científicamente el número óptimo de clusters (k).

In [ ]:
import pandas as pd
import numpy as np
import pyodbc
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

import warnings
warnings.filterwarnings('ignore')

### 1. Extracción de Datos desde SQL Server

In [ ]:
SERVER_NAME = 'LUIS'
DATABASE_NAME = 'DB_Analitica_Predictiva'
conn_str = f"DRIVER={{ODBC Driver 17 for SQL Server}};SERVER={SERVER_NAME};DATABASE={DATABASE_NAME};Trusted_Connection=yes;"

# Extraemos la Data con un JOIN SQL para obtener las etiquetas legibles
query = """
    SELECT 
        g.Provincia,
        s.NombreSector,
        e.NombreEmpresa,
        d.EnergiaFacturada_MWh,
        d.DemandaMaxima_MW,
        d.PorcentajePerdidas
    FROM Fact_Demanda d
    INNER JOIN Dim_Geografia g ON d.IdGeografia = g.IdGeografia
    INNER JOIN Dim_Sector s ON d.IdSector = s.IdSector
    INNER JOIN Dim_EmpresaElectrica e ON d.IdEmpresa = e.IdEmpresa
"""

conn = pyodbc.connect(conn_str)
df_ml = pd.read_sql(query, conn)
conn.close()

print(f"Datos extraídos: {df_ml.shape[0]} registros.")
df_ml.head()

### 2. Preprocesamiento (Escalamiento)
Los algoritmos basados en distancias (como K-Means) requieren que las variables numéricas estén en la misma escala.

In [ ]:
# Seleccionamos solo las variables numéricas (features) para encontrar los patrones de eficiencia
features = ['EnergiaFacturada_MWh', 'DemandaMaxima_MW', 'PorcentajePerdidas']
X = df_ml[features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Datos escalados correctamente. Ejemplo del primer registro:", X_scaled[0])

### 3. Método del Codo (Elbow Method) y Silueta
Entrenamos K-Means con diferentes valores de `k` (de 2 a 10) para encontrar cuántos grupos existen naturalmente en los datos.

In [ ]:
inercia = []
silueta = []
K_RANGO = range(2, 8) # Evaluaremos de 2 a 7 clusters

for k in K_RANGO:
    kmeans = KMeans(n_clusters=k, init='k-means++', random_state=42)
    kmeans.fit(X_scaled)
    
    inercia.append(kmeans.inertia_)
    # Evaluamos la métrica de silueta solo para 5000 muestras para no saturar memoria
    score = silhouette_score(X_scaled, kmeans.labels_, sample_size=5000, random_state=42)
    silueta.append(score)

# Generar Gráficas
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

ax1.plot(K_RANGO, inercia, marker='o', color='blue', linestyle='--')
ax1.set_title('Método del Codo (Inercia)')
ax1.set_xlabel('Número de Clusters (k)')
ax1.set_ylabel('Inercia')
ax1.grid(True)

ax2.plot(K_RANGO, silueta, marker='s', color='green', linestyle='-')
ax2.set_title('Coeficiente de Silueta')
ax2.set_xlabel('Número de Clusters (k)')
ax2.set_ylabel('Silueta Score')
ax2.grid(True)

plt.tight_layout()
plt.show()

### 4. Modelo Final y Visualización de Clusters
En base a las gráficas, el punto óptimo suele ser donde el codo se flexiona o la silueta es más alta (generalmente k=3 o k=4). Aplicaremos K-Means con `k=3` para catalogar las operaciones en 3 niveles (Ej. Eficiente, Regular, Ineficiente).

In [ ]:
# Modelo Definitivo
K_OPTIMO = 3
modelo_final = KMeans(n_clusters=K_OPTIMO, init='k-means++', random_state=42)
df_ml['Cluster'] = modelo_final.fit_predict(X_scaled)

print("Métricas Finales:")
print(f"- Inercia Final: {modelo_final.inertia_:.2f}")
print(f"- Iteraciones: {modelo_final.n_iter_}")
print(f"- Coeficiente Silueta: {silhouette_score(X_scaled, df_ml['Cluster'], sample_size=5000, random_state=42):.4f}")

# Renombrar clusters para contexto de negocio
cluster_names = {0: 'Operación Normal', 1: 'Alta Demanda', 2: 'Alta Ineficiencia (Pérdidas)'}
df_ml['Perfil'] = df_ml['Cluster'].map(cluster_names)

# Gráfica de Dispersión
plt.figure(figsize=(10, 6))
sns.scatterplot(x='DemandaMaxima_MW', y='PorcentajePerdidas', hue='Perfil', data=df_ml, palette='Set1', alpha=0.6)
plt.title('Agrupamiento K-Means: Demanda Máxima vs Pérdidas')
plt.xlabel('Demanda Máxima (MW)')
plt.ylabel('% Pérdidas de Energía')
plt.legend(title='Grupo')
plt.grid(True)
plt.show()

### Análisis de los Resultados (Impacto de Negocio)
Al graficar la **Demanda Máxima vs Porcentaje de Pérdidas**, el modelo descubrió automáticamente a qué grupo pertenecen las provincias o sectores. Podremos enfocar políticas públicas de ahorro de energía en los grupos marcados como "Alta Ineficiencia".